In [143]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from functools import reduce
import os
import sqlalchemy
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore') # supress the warnings

load_dotenv(dotenv_path=".env")

user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST")
database = os.getenv("MYSQL_DATABASE")

engine = sqlalchemy.create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:3306/{database}"
)

conn = engine.connect()

pd.read_sql("SHOW TABLES;", conn)

,Tables_in_ca2_agriculture


# Datasets Inspection & Ingestion 

## Milk Production Dataset

First, lets analyze the total production of milk available on the farm of each EU country.

In [144]:
df_milk_prod =  pd.read_csv("data/raw/milk_production.csv")
df_milk_prod.head()

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2015,3120.58,NaN,NaN
1,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2016,3114.43,NaN,NaN
2,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2017,3208.80,NaN,NaN
3,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2018,3203.83,NaN,NaN
4,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2019,3165.87,NaN,NaN


In [145]:
df_milk_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1939 entries, 0 to 1938
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1939 non-null   object 
 1   LAST UPDATE  1939 non-null   object 
 2   freq         1939 non-null   object 
 3   dairyprod    1939 non-null   object 
 4   milkitem     1939 non-null   object 
 5   geo          1939 non-null   object 
 6   TIME_PERIOD  1939 non-null   int64  
 7   OBS_VALUE    1913 non-null   float64
 8   OBS_FLAG     182 non-null    object 
 9   CONF_STATUS  26 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 151.6+ KB


In [146]:
df_milk_prod.describe(include='object')

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,OBS_FLAG,CONF_STATUS
count,1939,1939,1939,1939,1939,1939,182,26
unique,1,1,1,5,2,37,5,1
top,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,"Raw milk, total available on farms",Utilization of whole milk (1 000 t),Austria,p,C
freq,1939,1939,1939,660,1301,60,133,26


In [147]:
df_milk_prod["dairyprod"].unique()

array(['Farm milk products delivered to dairies',
       'Milk products, other than drinking milk, cream, butter and cheese, delivered to dairies',
       'Milk products, other than milk and cream, delivered to dairies',
       'Raw milk, total available on farms',
       'Raw milk delivered to dairies'], dtype=object)

In [148]:
df_milk_prod.milkitem.unique()

array(['Utilization of whole milk (1 000 t)',
       'Products obtained (1 000 t)'], dtype=object)

In [149]:
milk_prod = df_milk_prod[df_milk_prod["milkitem"] == "Products obtained (1 000 t)"][["geo","TIME_PERIOD","dairyprod","OBS_VALUE"]].reset_index(drop=True)
milk_prod.head()

,geo,TIME_PERIOD,dairyprod,OBS_VALUE
0,Austria,2015,"Milk products, other than milk and cream, deli...",0.0
1,Austria,2016,"Milk products, other than milk and cream, deli...",0.0
2,Austria,2017,"Milk products, other than milk and cream, deli...",0.0
3,Austria,2018,"Milk products, other than milk and cream, deli...",0.0
4,Austria,2019,"Milk products, other than milk and cream, deli...",0.0


In [150]:
milk_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   geo          638 non-null    object 
 1   TIME_PERIOD  638 non-null    int64  
 2   dairyprod    638 non-null    object 
 3   OBS_VALUE    630 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 20.1+ KB


Let's pivot the dataframe to convert the required features from long format to wide format, with only the observed relevant variables.

In [151]:
milk_prod= milk_prod.pivot_table(index=["geo","TIME_PERIOD"],columns="dairyprod",values="OBS_VALUE").reset_index()
milk_prod.columns.name = None  # Resetting column index name
milk_prod

,geo,TIME_PERIOD,"Milk products, other than milk and cream, delivered to dairies","Raw milk, total available on farms"
0,Austria,2015,0.0,3568.90
1,Austria,2016,0.0,3659.96
2,Austria,2017,0.0,3747.78
3,Austria,2018,0.0,3859.99
4,Austria,2019,0.0,3820.04
...,...,...,...,...
323,United Kingdom,2015,0.0,15457.00
324,United Kingdom,2016,0.0,14938.00
325,United Kingdom,2017,NaN,15443.00
326,United Kingdom,2018,0.0,15488.11


In [152]:
milk_prod.describe()

,TIME_PERIOD,"Milk products, other than milk and cream, delivered to dairies","Raw milk, total available on farms"
count,328.000000,302.000000,328.000000
mean,2019.435976,2.507450,12941.395884
std,2.850321,40.726065,33666.835111
min,2015.000000,0.000000,0.000000
25%,2017.000000,0.000000,916.970000
50%,2019.000000,0.000000,2406.140000
75%,2022.000000,0.000000,8570.272500
max,2024.000000,707.500000,174013.610000


From the previous outputs, the feature "Milk products, other than milk and cream, delivered to dairies" seems to contain only nan or 0 values, lets analyze it. Lets see if it contains non null values for Ireland , in that case it can be kept for other countries as well.

In [153]:
((milk_prod["geo"] == "Ireland") & (milk_prod["Milk products, other than milk and cream, delivered to dairies"].fillna(0) != 0)).any()

np.False_

Since the output is false, the feature "Milk products, other than milk and cream, delivered to dairies" can be dropped, while the "Raw milk, total available on farms" can be renamed for better readability.

In [154]:
milk_prod = milk_prod.drop(columns=["Milk products, other than milk and cream, delivered to dairies"])
milk_prod = milk_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "Raw milk, total available on farms": "milk_prod_1000t"
})
milk_prod.head()

,country,year,milk_prod_1000t
0,Austria,2015,3568.90
1,Austria,2016,3659.96
2,Austria,2017,3747.78
3,Austria,2018,3859.99
4,Austria,2019,3820.04


In [155]:
# Lets store the cleaned data table to the database.
milk_prod.to_csv("data/processed/milk_prod.csv",index=False)

## Milk Delivery and Consumption Dataset

In [156]:
df_products_milk =  pd.read_csv("data/raw/milk_delivery_&_consumption.csv")
df_products_milk.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Fat content (t),Raw cows' milk delivered to dairies,Percentage,Cyprus,2024,4.07,NaN,NaN
1,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Fat content (t),Raw cows' milk delivered to dairies,Percentage,France,2024,4.11,p,NaN
2,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2017,57.36,p,NaN
3,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2018,64.39,NaN,NaN
4,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2019,56.82,NaN,NaN


In [157]:
df_products_milk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1725 entries, 0 to 1724
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1725 non-null   object 
 1   LAST UPDATE  1725 non-null   object 
 2   freq         1725 non-null   object 
 3   milkitem     1725 non-null   object 
 4   dairyprod    1725 non-null   object 
 5   unit         1725 non-null   object 
 6   geo          1725 non-null   object 
 7   TIME_PERIOD  1725 non-null   int64  
 8   OBS_VALUE    1488 non-null   float64
 9   OBS_FLAG     222 non-null    object 
 10  CONF_STATUS  237 non-null    object 
dtypes: float64(1), int64(1), object(9)
memory usage: 148.4+ KB


In [158]:
df_products_milk.unit.unique()

array(['Percentage', 'Thousand tonnes'], dtype=object)

In [159]:
df_products_milk.dairyprod.unique()

array(["Raw cows' milk delivered to dairies",
       'Raw cream delivered to dairies (in milk equivalent)',
       'Drinking milk', 'Cream for direct consumption',
       'Milk and cream powders, excluding skimmed milk powders'],
      dtype=object)

In [160]:
df_products_milk = df_products_milk[df_products_milk["unit"] == "Thousand tonnes"][["geo","TIME_PERIOD","dairyprod","OBS_VALUE"]].reset_index(drop=True)
df_products_milk.head()

,geo,TIME_PERIOD,dairyprod,OBS_VALUE
0,Albania,2017,Raw cows' milk delivered to dairies,57.36
1,Albania,2018,Raw cows' milk delivered to dairies,64.39
2,Albania,2019,Raw cows' milk delivered to dairies,56.82
3,Albania,2020,Raw cows' milk delivered to dairies,56.27
4,Albania,2021,Raw cows' milk delivered to dairies,58.94


Lets convert the relevant variables from long format to wide format.

In [161]:
df_products_milk = df_products_milk .pivot_table(index=["geo","TIME_PERIOD"],columns="dairyprod",values="OBS_VALUE").reset_index()
df_products_milk.columns.name = None  # Resetting column index name
df_products_milk

,geo,TIME_PERIOD,Cream for direct consumption,Drinking milk,"Milk and cream powders, excluding skimmed milk powders",Raw cows' milk delivered to dairies,Raw cream delivered to dairies (in milk equivalent)
0,Albania,2017,0.21,8.93,0.00,57.36,0.0
1,Albania,2018,0.46,13.47,0.00,64.39,0.0
2,Albania,2019,0.23,10.15,0.00,56.82,0.0
3,Albania,2020,0.29,14.48,0.00,56.27,0.0
4,Albania,2021,0.29,12.19,0.00,58.94,0.0
...,...,...,...,...,...,...,...
362,Türkiye,2025,38.44,1639.76,39.86,NaN,NaN
363,United Kingdom,2016,289.60,6631.60,NaN,14542.00,0.0
364,United Kingdom,2017,307.20,6910.70,16.90,15144.70,0.0
365,United Kingdom,2018,284.90,6783.19,19.40,15188.10,0.0


In [162]:
df_products_milk.describe()

,TIME_PERIOD,Cream for direct consumption,Drinking milk,"Milk and cream powders, excluding skimmed milk powders",Raw cows' milk delivered to dairies,Raw cream delivered to dairies (in milk equivalent)
count,367.000000,313.000000,319.000000,278.000000,352.000000,222.000000
mean,2020.313351,91.258850,887.486865,24.264820,10959.964858,30.918423
std,2.907952,137.655781,1295.957456,40.950148,29688.736421,156.433609
min,2016.000000,0.210000,2.120000,0.000000,22.100000,0.000000
25%,2018.000000,14.360000,108.150000,0.000000,648.545000,0.000000
50%,2020.000000,31.930000,426.700000,0.945000,1884.215000,0.000000
75%,2023.000000,74.220000,726.825000,29.915000,7449.480000,0.000000
max,2025.000000,585.990000,6910.700000,184.100000,157382.630000,1164.730000


Lets analyze again if all features have non null values for Ireland, otherwise they will be dropped.

In [163]:
(df_products_milk[df_products_milk["geo"] == "Ireland"].fillna(0) != 0).any()

geo                                                        True
TIME_PERIOD                                                True
Cream for direct consumption                               True
Drinking milk                                              True
Milk and cream powders, excluding skimmed milk powders    False
Raw cows' milk delivered to dairies                        True
Raw cream delivered to dairies (in milk equivalent)       False
dtype: bool

Lets drop the features that don't add value to our analysis.

In [164]:
df_products_milk = df_products_milk.drop(columns=[
        "Milk and cream powders, excluding skimmed milk powders",
        "Raw cream delivered to dairies (in milk equivalent)" ])

milk_products = df_products_milk.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "Cream for direct consumption": "cream_direct_consumption_1000t",
    "Drinking milk": "processed_drinking_milk_1000t",
    "Raw cows' milk delivered to dairies":"milk_delivered_to_dairies_1000t"
})

milk_products.head()

,country,year,cream_direct_consumption_1000t,processed_drinking_milk_1000t,milk_delivered_to_dairies_1000t
0,Albania,2017,0.21,8.93,57.36
1,Albania,2018,0.46,13.47,64.39
2,Albania,2019,0.23,10.15,56.82
3,Albania,2020,0.29,14.48,56.27
4,Albania,2021,0.29,12.19,58.94


In [165]:
milk_products.to_csv("data/processed/milk_products.csv", index=False)

## Butter Production Dataset

In [166]:
butter_prod =  pd.read_csv("data/raw/butter_production.csv")
butter_prod.head()

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2014,0.68,NaN,NaN
1,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2015,0.94,p,NaN
2,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2016,0.82,p,NaN
3,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2017,0.88,NaN,NaN
4,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2018,0.91,NaN,NaN


In [167]:
butter_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     406 non-null    object 
 1   LAST UPDATE  406 non-null    object 
 2   freq         406 non-null    object 
 3   dairyprod    406 non-null    object 
 4   milkitem     406 non-null    object 
 5   geo          406 non-null    object 
 6   TIME_PERIOD  406 non-null    int64  
 7   OBS_VALUE    371 non-null    float64
 8   OBS_FLAG     14 non-null     object 
 9   CONF_STATUS  35 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.8+ KB


In [168]:
butter_prod["milkitem"].unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [169]:
butter_prod["dairyprod"].unique()

array(['Butter, incl. dehydrated butter and ghee, and other fats and oils derived from milk; dairy spreads'],
      dtype=object)

In [170]:
butter_prod.describe()

,TIME_PERIOD,OBS_VALUE
count,406.000000,371.000000
mean,2018.480296,78.042345
std,3.410710,121.059653
min,2013.000000,0.000000
25%,2016.000000,4.365000
50%,2018.000000,27.980000
75%,2021.000000,93.550000
max,2024.000000,509.490000


In [171]:
butter_prod = butter_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
butter_prod = butter_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "butter_prod_1000t"
})
butter_prod.head()

,country,year,butter_prod_1000t
0,Albania,2014,0.68
1,Albania,2015,0.94
2,Albania,2016,0.82
3,Albania,2017,0.88
4,Albania,2018,0.91


In [172]:
butter_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            406 non-null    object 
 1   year               406 non-null    int64  
 2   butter_prod_1000t  371 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 9.6+ KB


In [173]:
butter_prod.to_csv("data/processed/butter_prod.csv", index=False)

## Cheese Production Dataset

In [174]:
cheese_prod =  pd.read_csv("data/raw/cheese_production.csv")
cheese_prod.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2014,11.94,NaN,NaN
1,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2015,13.50,p,NaN
2,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2016,14.30,p,NaN
3,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2017,14.70,NaN,NaN
4,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2018,14.60,NaN,NaN


In [175]:
cheese_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     404 non-null    object 
 1   LAST UPDATE  404 non-null    object 
 2   freq         404 non-null    object 
 3   milkitem     404 non-null    object 
 4   dairyprod    404 non-null    object 
 5   geo          404 non-null    object 
 6   TIME_PERIOD  404 non-null    int64  
 7   OBS_VALUE    361 non-null    float64
 8   OBS_FLAG     11 non-null     object 
 9   CONF_STATUS  43 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.7+ KB


In [176]:
cheese_prod.milkitem.unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [177]:
cheese_prod.describe()

,TIME_PERIOD,OBS_VALUE
count,404.000000,361.000000
mean,2018.490099,371.701357
std,3.411759,559.915740
min,2013.000000,0.800000
25%,2016.000000,51.570000
50%,2018.000000,102.510000
75%,2021.000000,452.000000
max,2024.000000,2433.320000


In [178]:
cheese_prod = cheese_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
cheese_prod = cheese_prod.rename(columns={
    "geo":"country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "cheese_prod_1000t"
})
cheese_prod.head()

,country,year,cheese_prod_1000t
0,Albania,2014,11.94
1,Albania,2015,13.50
2,Albania,2016,14.30
3,Albania,2017,14.70
4,Albania,2018,14.60


In [179]:
cheese_prod.to_csv("data/processed/cheese_prod.csv", index=False)

## Milk Powder Production Dataset

In [180]:
milk_powder_prod =  pd.read_csv("data/raw/milk_powders_production.csv")
milk_powder_prod.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2014,0.0,NaN,NaN
1,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2015,0.0,p,NaN
2,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2016,0.0,p,NaN
3,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2017,0.0,NaN,NaN
4,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2018,0.0,NaN,NaN


In [181]:
milk_powder_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     407 non-null    object 
 1   LAST UPDATE  407 non-null    object 
 2   freq         407 non-null    object 
 3   milkitem     407 non-null    object 
 4   dairyprod    407 non-null    object 
 5   geo          407 non-null    object 
 6   TIME_PERIOD  407 non-null    int64  
 7   OBS_VALUE    299 non-null    float64
 8   OBS_FLAG     15 non-null     object 
 9   CONF_STATUS  108 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.9+ KB


In [182]:
milk_powder_prod.milkitem.unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [183]:
milk_powder_prod= milk_powder_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
milk_powder_prod = milk_powder_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "milk_powder_prod_1000t"
})
milk_powder_prod

,country,year,milk_powder_prod_1000t
0,Albania,2014,0.00
1,Albania,2015,0.00
2,Albania,2016,0.00
3,Albania,2017,0.00
4,Albania,2018,0.00
...,...,...,...
402,United Kingdom,2015,174.37
403,United Kingdom,2016,NaN
404,United Kingdom,2017,NaN
405,United Kingdom,2018,96.49


In [184]:
milk_powder_prod.describe()

,year,milk_powder_prod_1000t
count,407.000000,299.000000
mean,2018.454545,144.346622
std,3.427974,333.769658
min,2013.000000,0.000000
25%,2015.500000,0.000000
50%,2018.000000,33.990000
75%,2021.000000,134.250000
max,2024.000000,3055.660000


In [185]:
milk_powder_prod.to_csv("data/processed/milk_powder_prod.csv",index=False)

## Bovine Population Dataset

In [186]:
bovine_pop = pd.read_csv("data/raw/bovine_pop.csv")
bovine_pop.head()

,DATAFLOW,LAST UPDATE,freq,month,animals,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2016,1932.53,NaN,NaN
1,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2017,1938.48,NaN,NaN
2,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2018,1906.82,NaN,NaN
3,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2019,1873.31,NaN,NaN
4,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2020,1844.34,NaN,NaN


In [187]:
bovine_pop.shape

(2747, 11)

In [188]:
bovine_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2747 entries, 0 to 2746
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     2747 non-null   object 
 1   LAST UPDATE  2747 non-null   object 
 2   freq         2747 non-null   object 
 3   month        2747 non-null   object 
 4   animals      2747 non-null   object 
 5   unit         2747 non-null   object 
 6   geo          2747 non-null   object 
 7   TIME_PERIOD  2747 non-null   int64  
 8   OBS_VALUE    2746 non-null   float64
 9   OBS_FLAG     190 non-null    object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 236.2+ KB


In [189]:
bovine_pop.unit.unique()

array(['Thousand heads (animals)'], dtype=object)

In [190]:
bovine_pop.animals.unique()

array(['Live bovine animals', 'Bovine animals, less than 1 year old',
       'Bovine animals, less than 1 year old, for slaughter',
       'Bovine animals, less than 1 year old, not for slaughter',
       'Bovine animals, 1 to less than 2 years old'], dtype=object)

In [191]:
bovine_pop.month.unique()

array(['May-June', 'November - December'], dtype=object)

In [192]:
bovine_pop = bovine_pop[(bovine_pop['animals'] == 'Live bovine animals') & (bovine_pop['month'] == 'May-June')]
bovine_pop = bovine_pop[['geo', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={
    'geo': 'country',
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'bovine_pop_1000'
})
bovine_pop.head()

,country,year,bovine_pop_1000
0,Austria,2016,1932.53
1,Austria,2017,1938.48
2,Austria,2018,1906.82
3,Austria,2019,1873.31
4,Austria,2020,1844.34


In [193]:
bovine_pop.describe()

,year,bovine_pop_1000
count,165.000000,165.000000
mean,2020.503030,5699.535515
std,2.937612,5347.454236
min,2016.000000,79.900000
25%,2018.000000,1583.000000
50%,2020.000000,3731.750000
75%,2023.000000,7221.200000
max,2025.000000,19559.300000


In [194]:
bovine_pop.to_csv("data/processed/bovine_pop.csv",index=False)

## Dairy Cow Population Dataset

In [195]:
dairy_cow_pop = pd.read_csv("data/raw/dairy_cows_pop.csv")
dairy_cow_pop.head()

,DATAFLOW,LAST UPDATE,freq,month,animals,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2014,357.78,NaN,NaN
1,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2015,357.52,NaN,NaN
2,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2016,353.05,NaN,NaN
3,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2017,346.41,NaN,NaN
4,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2018,339.94,NaN,NaN


In [196]:
dairy_cow_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     445 non-null    object 
 1   LAST UPDATE  445 non-null    object 
 2   freq         445 non-null    object 
 3   month        445 non-null    object 
 4   animals      445 non-null    object 
 5   unit         445 non-null    object 
 6   geo          445 non-null    object 
 7   TIME_PERIOD  445 non-null    int64  
 8   OBS_VALUE    445 non-null    float64
 9   OBS_FLAG     34 non-null     object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 38.4+ KB


In [197]:
dairy_cow_pop.unit.unique()

array(['Thousand heads (animals)'], dtype=object)

In [198]:
dairy_cow_pop = dairy_cow_pop[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
dairy_cow_pop = dairy_cow_pop.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "dairy_cow_pop_1000"
})
dairy_cow_pop.head()

,country,year,dairy_cow_pop_1000
0,Albania,2014,357.78
1,Albania,2015,357.52
2,Albania,2016,353.05
3,Albania,2017,346.41
4,Albania,2018,339.94


In [199]:
dairy_cow_pop.describe()

,year,dairy_cow_pop_1000
count,445.000000,445.000000
mean,2019.433708,1341.475416
std,3.437765,3444.513588
min,2014.000000,5.510000
25%,2016.000000,114.900000
50%,2019.000000,282.910000
75%,2022.000000,1056.700000
max,2025.000000,21651.740000


In [200]:
dairy_cow_pop.to_csv("data/processed/dairy_cow_pop.csv",index=False)

## Selling Price for Farmers Dataset

In [201]:
raw_milk_price = pd.read_csv("data/raw/selling_price_milk.csv")
raw_milk_price.head()

,DATAFLOW,LAST UPDATE,freq,currency,prod_ani,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2013,37.55,NaN,NaN
1,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2014,39.46,NaN,NaN
2,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2015,33.73,NaN,NaN
3,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2016,31.33,NaN,NaN
4,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2017,37.28,NaN,NaN


In [202]:
raw_milk_price.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     290 non-null    object 
 1   LAST UPDATE  290 non-null    object 
 2   freq         290 non-null    object 
 3   currency     290 non-null    object 
 4   prod_ani     290 non-null    object 
 5   geo          290 non-null    object 
 6   TIME_PERIOD  290 non-null    int64  
 7   OBS_VALUE    280 non-null    float64
 8   OBS_FLAG     8 non-null      object 
 9   CONF_STATUS  7 non-null      object 
dtypes: float64(1), int64(1), object(8)
memory usage: 22.8+ KB


In [203]:
raw_milk_price.describe(include = "object")

,DATAFLOW,LAST UPDATE,freq,currency,prod_ani,geo,OBS_FLAG,CONF_STATUS
count,290,290,290,290,290,290,8,7
unique,1,1,1,1,1,26,2,1
top,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,p,C
freq,290,290,290,290,290,12,5,7


In [204]:
raw_milk_price = raw_milk_price[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
raw_milk_price = raw_milk_price.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "farmers_selling_price_100kg"
})
raw_milk_price

,country,year,farmers_selling_price_100kg
0,Austria,2013,37.55
1,Austria,2014,39.46
2,Austria,2015,33.73
3,Austria,2016,31.33
4,Austria,2017,37.28
...,...,...,...
285,United Kingdom,2015,32.76
286,United Kingdom,2016,26.81
287,United Kingdom,2017,31.86
288,United Kingdom,2018,32.19


In [205]:
raw_milk_price.to_csv("data/processed/raw_milk_price.csv",index=False)

## Economic Accounts

In [206]:
economic_account = pd.read_csv("data/raw/economic_account.csv")
economic_account.head()

,DATAFLOW,LAST UPDATE,freq,am_item,indic_agr,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2016,7000.76,NaN,NaN
1,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2017,7441.70,NaN,NaN
2,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2018,7500.74,NaN,NaN
3,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2019,7590.30,NaN,NaN
4,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2020,7720.12,NaN,NaN


In [207]:
economic_account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4356 entries, 0 to 4355
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     4356 non-null   object 
 1   LAST UPDATE  4356 non-null   object 
 2   freq         4356 non-null   object 
 3   am_item      4356 non-null   object 
 4   indic_agr    4356 non-null   object 
 5   unit         4356 non-null   object 
 6   geo          4356 non-null   object 
 7   TIME_PERIOD  4356 non-null   int64  
 8   OBS_VALUE    4029 non-null   float64
 9   OBS_FLAG     759 non-null    object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 374.5+ KB


In [208]:
economic_account.describe(include="object")

,DATAFLOW,LAST UPDATE,freq,am_item,indic_agr,unit,geo,OBS_FLAG
count,4356,4356,4356,4356,4356,4356,4356,759
unique,1,1,1,1,4,3,37,2
top,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,e
freq,4356,4356,4356,4356,1089,1452,120,432


In [209]:
economic_account["indic_agr"].unique()

array(['Production value at basic price',
       'Production value at producer price', 'Subsidies on products',
       'Taxes on products'], dtype=object)

In [210]:
economic_account["unit"].unique()

array(['Million euro', 'Million units of national currency',
       'Million purchasing power standards (PPS)'], dtype=object)

In [211]:
economic_account = economic_account[economic_account["unit"] == "Million euro"][["geo","TIME_PERIOD","indic_agr","OBS_VALUE"]]
economic_account.head()

,geo,TIME_PERIOD,indic_agr,OBS_VALUE
0,Austria,2016,Production value at basic price,7000.76
1,Austria,2017,Production value at basic price,7441.70
2,Austria,2018,Production value at basic price,7500.74
3,Austria,2019,Production value at basic price,7590.30
4,Austria,2020,Production value at basic price,7720.12


In [212]:
economic_account = economic_account.pivot_table(index=["geo","TIME_PERIOD"],columns="indic_agr",values="OBS_VALUE").reset_index()
economic_account.columns.name = None  # Resetting column index name
economic_account.head()

,geo,TIME_PERIOD,Production value at basic price,Production value at producer price,Subsidies on products,Taxes on products
0,Austria,2016,7000.76,7002.95,5.80,7.99
1,Austria,2017,7441.70,7441.23,7.21,6.74
2,Austria,2018,7500.74,7505.15,3.74,8.14
3,Austria,2019,7590.30,7595.23,3.72,8.65
4,Austria,2020,7720.12,7721.69,6.73,8.30


In [213]:
economic_account.describe()

,TIME_PERIOD,Production value at basic price,Production value at producer price,Subsidies on products,Taxes on products
count,363.000000,363.000000,363.000000,343.000000,274.000000
mean,2020.443526,73764.369725,72961.067934,938.633994,110.777007
std,2.867995,149062.443203,147514.860424,1791.047693,210.356113
min,2016.000000,121.090000,121.090000,0.000000,0.000000
25%,2018.000000,2818.010000,2611.745000,40.555000,0.000000
50%,2020.000000,8864.290000,8785.030000,135.350000,0.000000
75%,2023.000000,43819.380000,43758.575000,547.865000,49.312500
max,2025.000000,581890.560000,575936.570000,7484.120000,730.330000


In [214]:
economic_account = economic_account.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "Production value at basic price": "agri_output_basic_price_million_eur",
    "Production value at producer price": "agri_output_producer_price_million_eur",
    "Subsidies on products": "agri_subsidies_million_eur",
    "Taxes on products": "agri_taxes_million_eur"
})
economic_account.head()

,country,year,agri_output_basic_price_million_eur,agri_output_producer_price_million_eur,agri_subsidies_million_eur,agri_taxes_million_eur
0,Austria,2016,7000.76,7002.95,5.80,7.99
1,Austria,2017,7441.70,7441.23,7.21,6.74
2,Austria,2018,7500.74,7505.15,3.74,8.14
3,Austria,2019,7590.30,7595.23,3.72,8.65
4,Austria,2020,7720.12,7721.69,6.73,8.30


In [215]:
economic_account.to_csv("data/processed/economic_account.csv",index=False)

## Labour Input (Annual Work Units) Dataset

In [216]:
labour_input_awu = pd.read_csv("data/raw/agriculture_labour_input.csv")
labour_input_awu.head()

,DATAFLOW,LAST UPDATE,freq,am_item,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2016,121.07,NaN,NaN
1,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2017,121.82,NaN,NaN
2,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2018,121.30,NaN,NaN
3,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2019,120.31,NaN,NaN
4,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2020,121.61,NaN,NaN


In [217]:
labour_input_awu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1086 entries, 0 to 1085
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1086 non-null   object 
 1   LAST UPDATE  1086 non-null   object 
 2   freq         1086 non-null   object 
 3   am_item      1086 non-null   object 
 4   unit         1086 non-null   object 
 5   geo          1086 non-null   object 
 6   TIME_PERIOD  1086 non-null   int64  
 7   OBS_VALUE    1086 non-null   float64
 8   OBS_FLAG     108 non-null    object 
 9   CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 85.0+ KB


In [218]:
labour_input_awu.describe(include="object")

,DATAFLOW,LAST UPDATE,freq,am_item,unit,geo,OBS_FLAG
count,1086,1086,1086,1086,1086,1086,108
unique,1,1,1,3,1,37,1
top,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,e
freq,1086,1086,1086,362,1086,30,108


In [219]:
labour_input_awu.am_item.unique()

array(['Total agricultural labour input',
       'Non salaried agricultural labour input',
       'Salaried agricultural labour input'], dtype=object)

In [220]:
labour_input_awu.unit.unique()

array(['Thousand annual working units (AWU)'], dtype=object)

In [221]:
labour_input_awu = labour_input_awu.pivot_table(index=["geo","TIME_PERIOD"],columns="am_item",values="OBS_VALUE").reset_index()
labour_input_awu.columns.name = None 
labour_input_awu  = labour_input_awu .rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "Non salaried agricultural labour input": "non_salaried_labour_awu",
    "Salaried agricultural labour input": "salaried_labour_awu",
    "Total agricultural labour input": "total_labour_awu"
})
labour_input_awu.head()

,country,year,non_salaried_labour_awu,salaried_labour_awu,total_labour_awu
0,Austria,2016,102.69,18.39,121.07
1,Austria,2017,102.58,19.24,121.82
2,Austria,2018,101.52,19.78,121.30
3,Austria,2019,99.87,20.44,120.31
4,Austria,2020,101.64,19.97,121.61


In [222]:
labour_input_awu.to_csv("data/processed/labour_input_awu.csv",index=False)

## FAO Trade Dataset

In [223]:
trade_dairy_products = pd.read_csv("data/raw/FAO_trade_europe/Trade_CropsLivestock_E_Europe_NOFLAG.csv")
trade_dairy_products.head()

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Unit,Y1961,...,Y2015,Y2016,Y2017,Y2018,Y2019,Y2020,Y2021,Y2022,Y2023,Y2024
0,3,'008,Albania,221,'01371,"Almonds, in shell",5610,Import quantity,t,NaN,...,2.18,NaN,NaN,2.25,11.03,47.89,185.54,31.03,196.23,303.86
1,3,'008,Albania,221,'01371,"Almonds, in shell",5622,Import value,1000 USD,NaN,...,16.00,NaN,NaN,8.00,44.00,216.00,604.00,160.00,409.00,780.00
2,3,'008,Albania,221,'01371,"Almonds, in shell",5910,Export quantity,t,NaN,...,NaN,NaN,NaN,NaN,NaN,660.54,560.26,NaN,487.22,122.35
3,3,'008,Albania,221,'01371,"Almonds, in shell",5922,Export value,1000 USD,NaN,...,NaN,NaN,NaN,NaN,NaN,829.00,1400.00,NaN,846.00,435.00
4,3,'008,Albania,231,'21422,"Almonds, shelled",5610,Import quantity,t,NaN,...,133.22,128.62,58.59,84.56,287.25,367.18,351.63,1037.63,604.12,652.00


In [224]:
trade_dairy_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78410 entries, 0 to 78409
Data columns (total 73 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area Code        78410 non-null  int64  
 1   Area Code (M49)  78410 non-null  object 
 2   Area             78410 non-null  object 
 3   Item Code        78410 non-null  int64  
 4   Item Code (CPC)  78410 non-null  object 
 5   Item             78410 non-null  object 
 6   Element Code     78410 non-null  int64  
 7   Element          78410 non-null  object 
 8   Unit             78410 non-null  object 
 9   Y1961            33258 non-null  float64
 10  Y1962            33257 non-null  float64
 11  Y1963            33257 non-null  float64
 12  Y1964            33258 non-null  float64
 13  Y1965            33257 non-null  float64
 14  Y1966            33257 non-null  float64
 15  Y1967            33255 non-null  float64
 16  Y1968            33254 non-null  float64
 17  Y1969       

In [225]:
trade_dairy_products["Item"].head(10)

0              Almonds, in shell
1              Almonds, in shell
2              Almonds, in shell
3              Almonds, in shell
4               Almonds, shelled
5               Almonds, shelled
6               Almonds, shelled
7               Almonds, shelled
8    Animal oils and fats n.e.c.
9    Animal oils and fats n.e.c.
Name: Item, dtype: object

Filtering the required items as needed while referring the downloaded "Trade_CropsLivestock_E_Elements.csv" reference file.

In [226]:
target_items = ["Raw milk of cattle","Butter of cow milk","Cheese from whole cow milk","Whole milk powder"]
year_cols = [col for col in trade_dairy_products.columns if col.startswith("Y")] # As seen above all years are starting with "Y"

filtered_df = trade_dairy_products[trade_dairy_products["Item"].isin(target_items)] # Filtering for required rows

# Making a list of required features
columns_to_keep = [
    "Area",
    "Item",
    "Element",
    "Unit"
] + year_cols

filtered_df = filtered_df[columns_to_keep] # Filtering for required features
filtered_df.head()

,Area,Item,Element,Unit,Y1961,Y1962,Y1963,Y1964,Y1965,Y1966,...,Y2015,Y2016,Y2017,Y2018,Y2019,Y2020,Y2021,Y2022,Y2023,Y2024
134,Albania,Butter of cow milk,Import quantity,t,0.0,0.0,0.0,0.0,0.0,0.0,...,54.82,59.67,40.87,32.02,49.36,68.44,54.90,148.56,117.28,174.23
135,Albania,Butter of cow milk,Import value,1000 USD,0.0,0.0,0.0,0.0,0.0,0.0,...,244.00,261.00,270.00,235.00,311.00,418.00,369.00,1088.00,872.00,1421.00
136,Albania,Butter of cow milk,Export quantity,t,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137,Albania,Butter of cow milk,Export value,1000 USD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,Albania,Cheese from whole cow milk,Import quantity,t,0.0,0.0,0.0,0.0,0.0,0.0,...,131.19,194.53,222.66,245.23,184.35,2438.72,1050.95,3452.78,3376.62,4671.04


In [227]:
# Now to convert from wide format to long format, "melt" can be used.
tidy_df = filtered_df.melt(
    id_vars=["Area", "Item", "Element", "Unit"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

tidy_df.head()

,Area,Item,Element,Unit,Year,Value
0,Albania,Butter of cow milk,Import quantity,t,Y1961,0.0
1,Albania,Butter of cow milk,Import value,1000 USD,Y1961,0.0
2,Albania,Butter of cow milk,Export quantity,t,Y1961,NaN
3,Albania,Butter of cow milk,Export value,1000 USD,Y1961,NaN
4,Albania,Cheese from whole cow milk,Import quantity,t,Y1961,0.0


Now to get required features held by Item+Element columns, pivot can be applied but before lets concatenate the features along with their units.

In [228]:
# Integrating required feature sets together
tidy_df["feature"] = (
    tidy_df["Item"] + "_" +
    tidy_df["Element"] + "_" +
    tidy_df["Unit"]
)

df_trade= tidy_df.pivot_table(
    index=["Area", "Year"],
    columns="feature",
    values="Value",
    aggfunc="sum"
).reset_index()

df_trade.columns.name = None

df_trade.head()

,Area,Year,Butter of cow milk_Export quantity_t,Butter of cow milk_Export value_1000 USD,Butter of cow milk_Import quantity_t,Butter of cow milk_Import value_1000 USD,Cheese from whole cow milk_Export quantity_t,Cheese from whole cow milk_Export value_1000 USD,Cheese from whole cow milk_Import quantity_t,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD
0,Albania,Y1961,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Albania,Y1962,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Albania,Y1963,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Albania,Y1964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Albania,Y1965,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Lets clean up the features' names and year's observations.

In [229]:
df_trade["Year"] = df_trade["Year"].str.replace("Y", "").astype(int)
df_trade = df_trade.rename(columns={
    "Area": "country",
    "Year": "year"
})
df_trade.head()

,country,year,Butter of cow milk_Export quantity_t,Butter of cow milk_Export value_1000 USD,Butter of cow milk_Import quantity_t,Butter of cow milk_Import value_1000 USD,Cheese from whole cow milk_Export quantity_t,Cheese from whole cow milk_Export value_1000 USD,Cheese from whole cow milk_Import quantity_t,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD
0,Albania,1961,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Albania,1962,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Albania,1963,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Albania,1964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Albania,1965,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [230]:
df_trade.to_csv("data/processed/df_trade.csv",index=False)

## CAP Direct Payments Dataset

In [231]:
df_cap = pd.read_csv("data/raw/CAP_dataset/Indicators Financing.csv")
df_cap.head()

,Indicator Type,Indicator Category,Indicator Name,Sub-indicator Name,Sub-indicator Parameter,Sub-indicator Unit,Indicator Code,Member State Code,Member State Name,Time Period,Data,Flag
0,Output_Pillar I,Direct payments,OID_00 Direct payments (DP),Expenditure coupled payments,Total,billion euro,OID_00_3a,EU,European Union,2008,6.256,NaN
1,Output_Pillar I,Direct payments,OID_00 Direct payments (DP),Expenditure coupled payments,Total,billion euro,OID_00_3a,EU,European Union,2009,5.811,NaN
2,Output_Pillar I,Direct payments,OID_00 Direct payments (DP),Expenditure coupled payments,Total,billion euro,OID_00_3a,EU,European Union,2022,4.771,NaN
3,Output_Pillar I,Direct payments,OID_00 Direct payments (DP),Expenditure coupled payments,Total,billion euro,OID_00_3a,EU,European Union,2012,2.790,NaN
4,Output_Pillar I,Direct payments,OID_00 Direct payments (DP),Expenditure coupled payments,Total,billion euro,OID_00_3a,EU,European Union,2013,2.683,NaN


In [232]:
df_cap.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6393 entries, 0 to 6392
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Indicator Type           6393 non-null   object 
 1   Indicator Category       6393 non-null   object 
 2   Indicator Name           6393 non-null   object 
 3   Sub-indicator Name       6393 non-null   object 
 4   Sub-indicator Parameter  6393 non-null   object 
 5   Sub-indicator Unit       6393 non-null   object 
 6   Indicator Code           6393 non-null   object 
 7   Member State Code        6393 non-null   object 
 8   Member State Name        6393 non-null   object 
 9   Time Period              6393 non-null   int64  
 10  Data                     6393 non-null   float64
 11  Flag                     114 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 599.5+ KB


In [233]:
df_cap.describe(include="object")

,Indicator Type,Indicator Category,Indicator Name,Sub-indicator Name,Sub-indicator Parameter,Sub-indicator Unit,Indicator Code,Member State Code,Member State Name,Flag
count,6393,6393,6393,6393,6393,6393,6393,6393,6393,114
unique,2,4,14,30,13,3,39,28,28,2
top,Output_Pillar I,Rural Development,OIR_01 Total public expenditure,Market measures,Total,euro,OID_00_4,EU,European Union,F
freq,3585,2808,2536,1795,2042,6257,252,443,443,100


Lets analyze the unique values of required features.

In [234]:
cols_to_check = [
    'Indicator Category',
    'Indicator Name',
    'Sub-indicator Name',
    'Sub-indicator Unit',
    'Member State Name'
]

for col in cols_to_check:
    print(f"\nCOLUMN: {col}")
    print(df_cap[col].unique())


COLUMN: Indicator Category
['Direct payments' 'Horizontal aspects' 'Market measures'
 'Rural Development']

COLUMN: Indicator Name
['OID_00 Direct payments (DP)' 'OID_01 Basic Payment Scheme (BPS)'
 'OID_02 Single Area Payment Scheme (SAPS)'
 'OID_04 Redistributive payment' 'OID_05 Greening'
 'OID_12 Payment for young farmers' "OID_13 Small farmers' scheme"
 'OID_14 Voluntary coupled support'
 'OID_15 Payment for areas with natural constraints'
 'OID_16 Crop specific payment for cotton' 'OIH_00 Total CAP Expenditure'
 'OIM_00 Market Expenditure' 'OIR_00 Total Rural Development expenditure'
 'OIR_01 Total public expenditure']

COLUMN: Sub-indicator Name
['Expenditure coupled payments' 'Expenditure decoupled payments'
 'Expenditure direct payments' 'Expenditure Basic Payment Scheme'
 'Expenditure Single Area Payment Scheme'
 'Expenditure redistributive payment' 'Expenditure greening'
 'Expenditure young farmers' 'Expenditure small farmers'
 'Expenditure voluntary coupled support'
 'Expe

In [235]:
df_cap = df_cap[
    (df_cap['Indicator Name'] == 'OID_00 Direct payments (DP)') &
    (df_cap['Sub-indicator Unit'] == 'euro') &
    (df_cap['Member State Name'] != 'European Union')
][["Member State Name","Time Period","Data"]].reset_index(drop=True)
df_cap = df_cap.rename(columns={
    "Member State Name": "country",
    "Time Period": "year",
    "Data": "cap_direct_payments_eur"
})
df_cap.head()

,country,year,cap_direct_payments_eur
0,Bulgaria,2020,8.431556e+08
1,Croatia,2021,3.648424e+08
2,France,2019,6.909823e+09
3,Finland,2015,5.221946e+08
4,Czechia,2022,8.545906e+08


In [236]:
df_cap.to_csv("data/processed/df_cap.csv",index=False)

## Datasets Integration

Lets merge the collected datasets for further analysis using "merge" function. 

In [260]:
# First, lets create a list of all dataframes
dfs = [
    milk_prod,
    butter_prod,
    cheese_prod,
    milk_powder_prod,
    dairy_cow_pop,
    bovine_pop,
    raw_milk_price,
    milk_products,
    labour_input_awu,
    economic_account,
    df_trade,
    df_cap
]

# Lets clean up the joining keys
for df in dfs:
    df['country'] = df['country'].str.strip()
    df['year'] = df['year'].astype(int)
    
# https://docs.python.org/3/library/functools.html 
# https://www.geeksforgeeks.org/python/reduce-in-python/

# Can apply merge function using reduce() function
master_df = reduce(
    lambda left, right: pd.merge(left, right, on=['country', 'year'], how='outer'),
    dfs
)

master_df.head()

,country,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,cream_direct_consumption_1000t,...,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD,cap_direct_payments_eur
0,Albania,1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,Albania,1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,Albania,1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,Albania,1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,Albania,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


From the articles below the big players of the dairy industry have been identified to be : Germany, France, Netherlands, Poland and Italy. So, we will limit the master dataset to those countries only to be compared with Ireland.
1. https://www.europarl.europa.eu/meetdocs/2024_2029/plmrep/COMMITTEES/AGRI/DV/2026/01-12/EPRS_Briefing_The_EU_dairy_sector_EN.pdf 
2. https://www.farmersjournal.ie/news/news/who-are-the-big-players-in-the-eu-dairy-industry-663914
3. https://ec.europa.eu/eurostat/web/products-eurostat-news/w/ddn-20251120-3

In [263]:
countries_to_compare = ["Germany","France", "Netherlands","Poland","Italy","Ireland"]
master_df = master_df[master_df["country"].isin(countries_to_compare)].reset_index(drop=True)
master_df.head()

,country,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,cream_direct_consumption_1000t,...,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD,cap_direct_payments_eur
0,France,1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11016.0,71096.0,5633.0,6.0,2.0,10618.0,7615.0,3280.0,3462.0,NaN
1,France,1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18550.0,63444.0,5127.0,93.0,6.0,8857.0,5739.0,3195.0,3444.0,NaN
2,France,1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18868.0,67004.0,5609.0,67.0,3.0,9603.0,7147.0,5267.0,4739.0,NaN
3,France,1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22885.0,85665.0,7312.0,1467.0,256.0,13889.0,9871.0,12837.0,7343.0,NaN
4,France,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,35263.0,132358.0,12072.0,1.0,2.0,13083.0,11444.0,2807.0,3263.0,NaN


In [264]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 36 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   country                                           338 non-null    object 
 1   year                                              338 non-null    int64  
 2   milk_prod_1000t                                   60 non-null     float64
 3   butter_prod_1000t                                 69 non-null     float64
 4   cheese_prod_1000t                                 72 non-null     float64
 5   milk_powder_prod_1000t                            61 non-null     float64
 6   dairy_cow_pop_1000                                72 non-null     float64
 7   bovine_pop_1000                                   60 non-null     float64
 8   farmers_selling_price_100kg                       64 non-null     float64
 9   cream_direct_consumpt

In [265]:
master_df.shape

(338, 36)

In [280]:
master_df.duplicated(subset=['country', 'year']).sum()

np.int64(0)

As seen the master dataset contains values starting from 1961 so for eda we will keep our dataset starting from the year where all the values of Ireland are present. 

In [275]:
df_ie = master_df[master_df['country'] == 'Ireland']

# Finding rows with no Nan values at all
full_row = ireland_df[ireland_df.notna().all(axis=1)]
print(full_row['year'].iloc[0])

2016


In [276]:
eda_df = master_df[(master_df['year'] >= 2016)].reset_index(drop=True)
eda_df.head()

,country,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,cream_direct_consumption_1000t,...,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD,cap_direct_payments_eur
0,France,2016,26065.89,434.23,1918.97,516.27,3637.02,19559.30,31.1,478.85,...,1377085.0,827774.68,394645.0,267741.96,161375.0,85313.31,259515.0,35739.69,95770.0,7.365412e+09
1,France,2017,26006.31,412.72,1919.57,504.57,3596.84,19432.72,NaN,502.89,...,1602319.0,695073.17,387469.0,220518.76,167175.0,81144.07,295481.0,36003.52,107764.0,7.193522e+09
2,France,2018,26022.50,417.41,1907.76,451.82,3554.23,18736.99,NaN,503.07,...,1752015.0,713772.41,384316.0,162278.98,139781.0,73302.50,274410.0,39473.85,117547.0,6.934972e+09
3,France,2019,26036.29,419.22,1903.29,470.06,3490.81,18469.77,NaN,514.13,...,1771316.0,606901.07,331104.0,165966.35,141125.0,68567.23,253199.0,45350.27,131657.0,6.909823e+09
4,France,2020,26288.53,417.54,1862.10,486.48,3405.68,18200.48,NaN,535.13,...,1937528.0,599534.18,328154.0,135087.36,122292.0,76865.78,290181.0,36569.60,117665.0,6.807766e+09


In [293]:
dairy_df = eda_df[
    [
        'country',
        'year',
        'milk_prod_1000t',
        'butter_prod_1000t',
        'cheese_prod_1000t',
        'milk_powder_prod_1000t',
        'dairy_cow_pop_1000',
        'bovine_pop_1000',
        'farmers_selling_price_100kg',
        'milk_delivered_to_dairies_1000t',
        'cream_direct_consumption_1000t',
        'processed_drinking_milk_1000t'
    ]
]

dairy_df.to_csv("data/processed/dairy_df.csv", index=False)

dairy_df.head()

,country,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,milk_delivered_to_dairies_1000t,cream_direct_consumption_1000t,processed_drinking_milk_1000t
0,France,2016,26065.89,434.23,1918.97,516.27,3637.02,19559.30,31.1,24453.33,478.85,3294.44
1,France,2017,26006.31,412.72,1919.57,504.57,3596.84,19432.72,NaN,24629.49,502.89,3229.51
2,France,2018,26022.50,417.41,1907.76,451.82,3554.23,18736.99,NaN,24542.54,503.07,3102.61
3,France,2019,26036.29,419.22,1903.29,470.06,3490.81,18469.77,NaN,24526.30,514.13,3007.98
4,France,2020,26288.53,417.54,1862.10,486.48,3405.68,18200.48,NaN,24602.21,535.13,3067.96


In [294]:
dairy_df.isna().sum()

country                             0
year                                0
milk_prod_1000t                     6
butter_prod_1000t                   8
cheese_prod_1000t                   6
milk_powder_prod_1000t             12
dairy_cow_pop_1000                  0
bovine_pop_1000                     0
farmers_selling_price_100kg        14
milk_delivered_to_dairies_1000t     0
cream_direct_consumption_1000t     12
processed_drinking_milk_1000t      10
dtype: int64

In [295]:
dairy_df.describe()

,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,milk_delivered_to_dairies_1000t,cream_direct_consumption_1000t,processed_drinking_milk_1000t
count,60.00000,54.000000,52.000000,54.000000,48.000000,60.000000,60.000000,46.000000,60.000000,48.000000,50.000000
mean,2020.50000,18356.863704,290.066731,1277.370556,407.322083,2376.379333,8804.501333,39.817391,17256.711667,313.133542,2428.416400
std,2.89652,8489.043457,131.442692,684.904753,199.590642,938.942316,4767.422335,8.737751,8361.778226,207.016482,1291.860063
min,2016.00000,6871.940000,91.220000,206.100000,25.760000,1295.230000,3551.840000,25.390000,6851.630000,16.410000,493.390000
25%,2018.00000,13352.452500,216.830000,894.335000,200.012500,1565.250000,6069.242500,33.595000,11934.775000,140.140000,1805.575000
50%,2020.50000,14827.935000,257.780000,1128.165000,436.680000,1997.740000,6674.455000,37.390000,13631.800000,263.120000,2468.500000
75%,2023.00000,25962.002500,413.900000,1907.587500,517.877500,3342.942500,11238.045000,46.002500,24266.985000,528.105000,3093.947500
max,2025.00000,34033.750000,506.930000,2433.320000,725.010000,4217.700000,19559.300000,58.230000,32548.980000,585.990000,4993.060000


In [296]:
dairy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 12 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   country                          60 non-null     object 
 1   year                             60 non-null     int64  
 2   milk_prod_1000t                  54 non-null     float64
 3   butter_prod_1000t                52 non-null     float64
 4   cheese_prod_1000t                54 non-null     float64
 5   milk_powder_prod_1000t           48 non-null     float64
 6   dairy_cow_pop_1000               60 non-null     float64
 7   bovine_pop_1000                  60 non-null     float64
 8   farmers_selling_price_100kg      46 non-null     float64
 9   milk_delivered_to_dairies_1000t  60 non-null     float64
 10  cream_direct_consumption_1000t   48 non-null     float64
 11  processed_drinking_milk_1000t    50 non-null     float64
dtypes: float64(10), int64(1)

In [297]:
country_summary = dairy_df.groupby("country").mean(numeric_only=True)

country_summary

,year,milk_prod_1000t,butter_prod_1000t,cheese_prod_1000t,milk_powder_prod_1000t,dairy_cow_pop_1000,bovine_pop_1000,farmers_selling_price_100kg,milk_delivered_to_dairies_1000t,cream_direct_consumption_1000t,processed_drinking_milk_1000t
country,,,,,,,,,,,
France,2020.5,25802.801111,414.008889,1903.622222,468.567778,3346.125,17990.520,31.100000,24239.423,534.0590,2903.931
Germany,2020.5,33053.605556,480.562222,2315.308889,692.844444,3899.197,11427.045,38.792222,32232.307,543.1300,4385.244
Ireland,2020.5,8287.183333,280.087500,260.077778,495.650000,1438.756,7264.071,38.740000,8349.710,23.3375,528.631
Italy,2020.5,13486.588889,94.306667,1323.265556,32.905000,1883.942,6053.232,45.305556,12306.759,145.1780,2449.042
Netherlands,2020.5,14830.391111,228.985000,960.876667,364.936667,1589.111,3824.854,40.480000,13913.323,NaN,NaN
Poland,2020.5,14680.612222,234.554444,901.072222,190.833333,2101.145,6267.286,36.737778,12498.748,262.0040,1875.234


In [290]:
economics_df = eda_df[
    [
        'country',
        'year',
        'agri_output_basic_price_million_eur',
        'agri_output_producer_price_million_eur',
        'agri_subsidies_million_eur',
        'agri_taxes_million_eur',
        'cap_direct_payments_eur'
    ]
]
economics_df.to_csv("data/processed/economics_df.csv", index=False)

economics_df.head()

,country,year,agri_output_basic_price_million_eur,agri_output_producer_price_million_eur,agri_subsidies_million_eur,agri_taxes_million_eur,cap_direct_payments_eur
0,France,2016,70485.70,69325.72,1177.61,17.62,7.365412e+09
1,France,2017,73152.38,72010.93,1158.49,17.04,7.193522e+09
2,France,2018,78139.89,77016.54,1139.10,15.76,6.934972e+09
3,France,2019,77792.69,76644.78,1147.91,0.00,6.909823e+09
4,France,2020,75101.33,73925.06,1176.26,0.00,6.807766e+09


In [291]:
labour_df = eda_df[
    [
        'country',
        'year',
        'non_salaried_labour_awu',
        'salaried_labour_awu',
        'total_labour_awu'
    ]
]

labour_df.to_csv("data/processed/labour_df.csv", index=False)

labour_df.head()

,country,year,non_salaried_labour_awu,salaried_labour_awu,total_labour_awu
0,France,2016,474.29,278.62,752.91
1,France,2017,464.81,280.85,745.66
2,France,2018,456.86,284.22,741.08
3,France,2019,446.71,288.20,734.92
4,France,2020,428.66,289.15,717.81


In [292]:
trade_accounts_df = eda_df[
    [
        'country',
        'year',

        # Butter Trade
        'Butter of cow milk_Export quantity_t',
        'Butter of cow milk_Export value_1000 USD',
        'Butter of cow milk_Import quantity_t',
        'Butter of cow milk_Import value_1000 USD',

        # Cheese Trade
        'Cheese from whole cow milk_Export quantity_t',
        'Cheese from whole cow milk_Export value_1000 USD',
        'Cheese from whole cow milk_Import quantity_t',
        'Cheese from whole cow milk_Import value_1000 USD',

        # Raw Milk Trade
        'Raw milk of cattle_Export quantity_t',
        'Raw milk of cattle_Export value_1000 USD',
        'Raw milk of cattle_Import quantity_t',
        'Raw milk of cattle_Import value_1000 USD',

        # Milk Powder Trade
        'Whole milk powder_Export quantity_t',
        'Whole milk powder_Export value_1000 USD',
        'Whole milk powder_Import quantity_t',
        'Whole milk powder_Import value_1000 USD'
    ]
]

trade_accounts_df.to_csv("data/processed/trade_df.csv", index=False)


trade_accounts_df.head()

,country,year,Butter of cow milk_Export quantity_t,Butter of cow milk_Export value_1000 USD,Butter of cow milk_Import quantity_t,Butter of cow milk_Import value_1000 USD,Cheese from whole cow milk_Export quantity_t,Cheese from whole cow milk_Export value_1000 USD,Cheese from whole cow milk_Import quantity_t,Cheese from whole cow milk_Import value_1000 USD,Raw milk of cattle_Export quantity_t,Raw milk of cattle_Export value_1000 USD,Raw milk of cattle_Import quantity_t,Raw milk of cattle_Import value_1000 USD,Whole milk powder_Export quantity_t,Whole milk powder_Export value_1000 USD,Whole milk powder_Import quantity_t,Whole milk powder_Import value_1000 USD
0,France,2016,74702.85,334064.0,174665.46,583959.0,601700.26,2917472.0,295365.18,1377085.0,827774.68,394645.0,267741.96,161375.0,85313.31,259515.0,35739.69,95770.0
1,France,2017,74807.40,440864.0,187257.55,974837.0,609827.27,3060918.0,313756.02,1602319.0,695073.17,387469.0,220518.76,167175.0,81144.07,295481.0,36003.52,107764.0
2,France,2018,72418.78,487249.0,194351.93,1144286.0,610659.00,3271157.0,328026.73,1752015.0,713772.41,384316.0,162278.98,139781.0,73302.50,274410.0,39473.85,117547.0
3,France,2019,73951.09,453927.0,195972.60,912672.0,615934.55,3117875.0,342807.35,1771316.0,606901.07,331104.0,165966.35,141125.0,68567.23,253199.0,45350.27,131657.0
4,France,2020,74614.45,439690.0,180900.36,727462.0,594873.13,3150066.0,364870.43,1937528.0,599534.18,328154.0,135087.36,122292.0,76865.78,290181.0,36569.60,117665.0
